## Prepare dataset in the required format

### image url

The met image urls are not publicly accessible. The images paths on google colab are not https public urls. So upload the images to github and then re-extract the public https urls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers huggingface_hub

In [ ]:
from huggingface_hub import HfApi, login

# Log in (paste HF token once)
login()

api = HfApi()

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/My Drive/master_thesis_project/met_metadata_0509.csv")
df.head()

,ID,Inventory Number,Title,Medium,Date,Standardized Date,Dimension,Period,Culture,Category,Description,Link Resource,Object Wikidata URL,Tags,Tags AAT URL,Bibliography
0,239585,74.51.3631,Pendant in the form of a vase,Gold,5th–4th century BCE,499-300 BCE,Other: 9/16 in. (1.4 cm),Classical,Greek,Gold and Silver,Gold pendant in the form of a vase.,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q116305682,NaN,NaN,"Myres, John L. 1914. Handbook of the Cesnola C..."
1,239586,74.51.1,Glass bead,Glass,ca. 330–70 BCE,330-70 BCE,L.: 1 in. (2.4 cm)\r\nDiam.: 3/8 in. (1.0 cm),Hellenistic,Greek,Glass,"Translucent deep purple, appearing black; trai...",http://www.metmuseum.org/art/collection/search...,NaN,NaN,NaN,"Cesnola, Luigi Palma di. 1903. A Descriptive A..."
2,239588,74.51.3,Glass eye bead,Glass,6th–4th century BCE,599-300 BCE,L.: 3/8 in. (1 cm)\r\nDiam.: 15/16 in. (0.9 cm),Archaic or Classical,"Phoenician, Cypriot",Glass,Stratified double eye bead; two sets of four e...,http://www.metmuseum.org/art/collection/search...,NaN,NaN,NaN,"Cesnola, Luigi Palma di. 1903. A Descriptive A..."
3,239593,74.51.8,Glass bead,Glass,ca. 330–70 BCE,330-70 BCE,L. 7/8 in. (2.2 cm)\r\ndiameter 7/16 in. (1.1...,Hellenistic,Greek,Glass,"Uncertain color, probably translucent deep pur...",http://www.metmuseum.org/art/collection/search...,NaN,NaN,NaN,"Myres, John L. 1914. Handbook of the Cesnola C..."
4,239596,74.51.11,Glass bead,Glass,ca. 750–300 BCE,750-300 BCE,L.: 15/16in. (2.5 cm)\r\nDiam.: 3/8in. (1 cm),Archaic or Classical,Cypriot,Glass,"Uncertain color, appearing black; trail in opa...",http://www.metmuseum.org/art/collection/search...,NaN,NaN,NaN,"Myres, John L. 1914. Handbook of the Cesnola C..."


In [ ]:
import json
import pandas as pd
import os

input_file = "/content/drive/My Drive/master_thesis_project/met_metadata_0509.csv"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only.jsonl"

# Hugging Face URL prefix
hf_prefix = "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/"

# Read CSV file
df = pd.read_csv(input_file)

# Get all image files from Hugging Face to map artifact IDs to available images
repo_files = list(api.list_repo_files(repo_id="Huan3/met_artefacts_images", repo_type="dataset"))
image_files = [f for f in repo_files if f.lower().endswith('.jpg')]

# Create a mapping from artifact ID to its available images
artifact_images = {}
for image_file in image_files:
    artifact_id = image_file.split('_')[0]  # Extract ID from filename
    if artifact_id not in artifact_images:
        artifact_images[artifact_id] = []
    artifact_images[artifact_id].append(image_file)

# Sort images so primary comes first, then additional_0, additional_1, etc.
def sort_images(images):
    def sort_key(img):
        if '_primary' in img:
            return (0, 0)
        elif '_additional_' in img:
            try:
                num = int(img.split('_additional_')[1].split('.')[0])
                return (1, num)
            except:
                return (2, img)
        else:
            return (3, img)
    return sorted(images, key=sort_key)

with open(output_file, "w", encoding="utf-8") as f_out:

    for _, row in df.iterrows():
        artifact_id = str(row.get("ID", "")).strip()

        material = row.get("Medium", "")
        culture = row.get("Culture", "")
        category = row.get("Category", "")

        # Skip if any label missing
        if not (material and culture and category):
            continue

        # Get available images for this artifact
        available_images = artifact_images.get(artifact_id, [])
        sorted_images = sort_images(available_images)[:10]  # Max 10 images

        # Skip if no images available
        if not sorted_images:
            continue

        # Create image URLs
        hf_images = []
        for img_filename in sorted_images:
            hf_images.append({
                "type": "image_url",
                "image_url": {"url": hf_prefix + img_filename}
            })

        # Build conversation with English prompts
        messages = [
            {
                "role": "system",
                "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."
            },
            {
                "role": "user",
                "content": hf_images + [
                    {"type": "text", "text": "Please provide the metadata (Material, Culture, Category) for this artifact."}
                ]
            },
            {
                "role": "assistant",
                "content": f"Material: {material}\nCulture: {culture}\nCategory: {category}"
            }
        ]

        entry = {"messages": messages}
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nDone! JSONL with HuggingFace image URLs written to {output_file}")


Done! JSONL with HuggingFace image URLs written to /content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only.jsonl


In [ ]:
# Show the first 5 lines of the JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip())

{"messages": [{"role": "system", "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239585_primary.jpg"}}, {"type": "text", "text": "Please provide the metadata (Material, Culture, Category) for this artifact."}]}, {"role": "assistant", "content": "Material: Gold\nCulture: Greek\nCategory: Gold and Silver"}]}
{"messages": [{"role": "system", "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239586_primary.jpg"}}, {"type": "text", "text": "Please provide the metadata (Material, Culture, Category) for this artifact."}]}, {"role": "assistant", "content": "Material: Glass\nCulture: Greek\nCategory: Glass"}]}

In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 0:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/239585_primary.jpg"
          }
        },
        {
          "type": "text",
          "text": "Please provide the metadata (Material, Culture, Category) for this artifact."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Material: Gold\nCulture: Greek\nCategory: Gold and Silver"
    }
  ]
}


Large images would be more tokens and cost more. According to https://platform.openai.com/docs/guides/vision-fine-tuning#size :
If you set the detail parameter for an image to low, the image is resized to 512 by 512 pixels and is only represented by 85 tokens regardless of its size. This will reduce the cost of training.

In [ ]:
import os
import json

jsonl_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only.jsonl"

# Check file size
file_size_mb = os.path.getsize(jsonl_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Check number of images per example
max_images_allowed = 10
supported_formats = (".jpg", ".jpeg", ".png", ".webp")
max_10_images = True

with open(jsonl_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            example = json.loads(line)
            user_content = example["messages"][1]["content"]
            if not isinstance(user_content, list):
                print(f"Example {i}: user content is not a list")
                continue

            # Extract image URLs
            image_urls = [x["image_url"]["url"] for x in user_content if x.get("type") == "image_url"]

            if len(image_urls) > max_images_allowed:
                print(f"Example {i}: has {len(image_urls)} images (consider reducing to {max_images_allowed})")
                max_10_images = False

            # check formats
            for url in image_urls:
                if not url.lower().endswith(supported_formats):
                    print(f"Example {i}: unsupported image format -> {url}")

        except Exception as e:
            print(f"Error processing example {i}: {e}")

if max_10_images:
    print(f"All the artefacts have maximal 10 images.")

File size: 3.29 MB
All the artefacts have maximal 10 images.


https://community.openai.com/t/gpt-4-vision-preview-fidelity-detail-parameter/477563

add the parameter detail=low into data  

e.g. taken from https://platform.openai.com/docs/guides/vision-fine-tuning#size  

{
  "type": "image_url",
  "image_url": {
    "url": "https://upload.wikimedia.org/wikipedia/commons/3/36/Danbo_Cheese.jpg",
    "detail": "low"
  }
}

In [ ]:
# add detail=low
import json

# Input and output paths
input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only.jsonl"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low.jsonl"

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        data = json.loads(line)

        # Go through messages
        for message in data.get("messages", []):
            if isinstance(message.get("content"), list):
                for item in message["content"]:
                    if (isinstance(item, dict) and
                        item.get("type") == "image_url" and
                        "image_url" in item):
                        item["image_url"]["detail"] = "low"

        # Write updated data
        outfile.write(json.dumps(data, ensure_ascii=False) + "\n")

print("All image entries updated with detail='low' and saved to:")
print(output_file)

All image entries updated with detail='low' and saved to:
/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low.jsonl


taking random 495 samples, so that as many as possible labels can be included

In [ ]:
import random

input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low.jsonl"
output_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low_495.jsonl"

count = 495

# Read all lines
with open(input_file, "r") as fin:
    lines = fin.readlines()

# Shuffle all lines and take first 'count'
random.shuffle(lines)
selected_lines = lines[:count]

# Write to output file
with open(output_file, "w") as fout:
    for line in selected_lines:
        fout.write(line)

print(f"Saved {len(selected_lines)} randomly shuffled lines to: {output_file}")

Saved 495 randomly shuffled lines to: /content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low_495.jsonl


In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(output_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 30:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255387_primary.jpg",
            "detail": "low"
          }
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255387_additional_0.jpg",
            "detail": "low"
          }
        },
        {
          "type": "text",
          "text": "Please provide the metadata (Material, Culture, Category) for this artifact."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Material: Bronze\nCulture: Etruscan\nCategory: Bronzes"
    }
  ]
}


### Split the data into train, val and test set in the ratio of 8:1:1

In [ ]:
import json
import random
from pathlib import Path

random.seed(42)  # For reproducible splits

input_file = "/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_classification_only_detail_low_495.jsonl"
output_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Load all examples
with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

total = len(lines)
print(f"Total examples: {total}")

# Shuffle the data
random.shuffle(lines)

# Compute split indices
train_end_idx = int(total * 0.8)
val_end_idx = int(total * 0.9)

train = lines[:train_end_idx]
val = lines[train_end_idx:val_end_idx]
test = lines[val_end_idx:]

# Save splits
splits = {"train": train, "validation": val, "test": test}

for split_name, split_data in splits.items():
    output_file = Path(output_dir) / f"{split_name}.jsonl"
    with open(output_file, "w", encoding="utf-8") as f_out:
        for line in split_data:
            f_out.write(line)
    print(f"{split_name}: {len(split_data)} examples saved to {output_file}")

# Count total images for each split file
print("\n=== Image Counts ===")
for split_name in ["train", "validation", "test"]:
    split_file = Path(output_dir) / f"{split_name}.jsonl"
    total_images = 0

    with open(split_file, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            for message in data.get("messages", []):
                content = message.get("content")
                if isinstance(content, list):
                    for item in content:
                        if isinstance(item, dict) and item.get("type") == "image_url":
                            total_images += 1

    print(f"{split_name}: {total_images} total images")

Total examples: 495
train: 396 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/train.jsonl
validation: 49 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/validation.jsonl
test: 50 examples saved to /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/test.jsonl

=== Image Counts ===
train: 626 total images
validation: 76 total images
test: 91 total images


In [ ]:
# make sure the encoding is utf-8 for gpt4o fine-tuning
import chardet

with open("/content/drive/My Drive/master_thesis_project/met_data_for_gpt4o_description_only_detail_low.jsonl", "rb") as f:
    raw = f.read(4096)
    print(chardet.detect(raw))

{'encoding': 'ascii', 'confidence': 1.0, 'language': ''}


### Fine-tuning

In [ ]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/948.4 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.107.0
    Uninstalling openai-1.107.0:
      Successfully uninstalled openai-1.107.0


#### load the inspect the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
met_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only"

In [ ]:
!wc -l "{met_data_dir}/train.jsonl"
!wc -l "{met_data_dir}/validation.jsonl"
!wc -l "{met_data_dir}/test.jsonl"

396 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/train.jsonl
49 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/validation.jsonl
50 /content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only/test.jsonl


In [ ]:
!head -n 10 "{met_data_dir}/train.jsonl"

{"messages": [{"role": "system", "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/241446_primary.jpg", "detail": "low"}}, {"type": "text", "text": "Please provide the metadata (Material, Culture, Category) for this artifact."}]}, {"role": "assistant", "content": "Material: Terracotta\nCulture: Cypriot\nCategory: Terracottas"}]}
{"messages": [{"role": "system", "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/246405_primary.jpg", "detail": "low"}}, {"type": "text", "text": "Please provide the metadata (Material, Culture, Category) for this artifact."}]}, {"role": "assistant", "content": "Material: Bron

In [ ]:
import json

# Show one pretty-printed sample from your JSONL file
with open(f"{met_data_dir}/train.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 1:  # just take the first artefact
            obj = json.loads(line)
            print(json.dumps(obj, indent=2, ensure_ascii=False))
            break

{
  "messages": [
    {
      "role": "system",
      "content": "You are a museum curator who creates accurate metadata for archaeological artifacts."
    },
    {
      "role": "user",
      "content": [
        {
          "type": "image_url",
          "image_url": {
            "url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/246405_primary.jpg",
            "detail": "low"
          }
        },
        {
          "type": "text",
          "text": "Please provide the metadata (Material, Culture, Category) for this artifact."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "Material: Bronze\nCulture: Italic\nCategory: Bronzes"
    }
  ]
}


#### run gpt4o fine-tuning

use the fine-tunable vision model gpt-4o-2024-08-06  
https://platform.openai.com/docs/guides/vision-fine-tuning . But in this link, it says that "Each example can have at most 10 images."

Accoding to https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/fine-tuning-vision :
Vision fine-tuning is supported for gpt-4o version 2024-08-06 and gpt-4.1 version 2025-04-14 models only. In this link, it says "Each example can have at most 64 images." This is written in 2025. but in this link https://platform.openai.com/docs/guides/supervised-fine-tuning gpt-4.1 version 2025-04-14 models are not listed in the vision fine-tuning page.

In [ ]:
# initiate openai client

from openai import OpenAI

client = OpenAI(api_key="sk-xxxx")

upload the train and validation files

In [ ]:
training_file_upload_response = client.files.create(
    file=open(f"{met_data_dir}/train.jsonl", "rb"),
    purpose="fine-tune"
)

validation_file_upload_response = client.files.create(
    file=open(f"{met_data_dir}/validation.jsonl", "rb"),
    purpose="fine-tune"
)

print("training_file_upload_response:", training_file_upload_response)
print("validation_file_upload_response:", validation_file_upload_response)

training_file_upload_response: FileObject(id='file-DEuQU4JbtLbg8CJDDedHbU', bytes=245732, created_at=1763466175, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
validation_file_upload_response: FileObject(id='file-TGr42f3AcZ2FnKNbsYZD5d', bytes=30219, created_at=1763466176, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


create a fine-tuned mode

In [ ]:
from datetime import datetime
import re

def generate_finetune_suffix(project_name: str, dataset_name: str, version: str = None) -> str:
    """
    Generates a kebab-case suffix for fine-tuning jobs, where spaces are replaced with hyphens and all letters are lower-case.

    Args:
        project_name (str): Short name for the project, e.g., 'artefacts'.
        dataset_name (str): Name of the dataset, e.g., 'uzh'.
        version (str, optional): Version or date, e.g., 'v1' or '2025-10-15'. Defaults to today.

    Returns:
        str: kebab-case suffix, e.g., 'artefacts-uzh-2025-10-15'
    """
    def to_kebab(text: str) -> str:
        return re.sub(r'\s+', '-', text.strip().lower())

    if version is None:
        version = datetime.today().strftime("%Y-%m-%d")

    parts = [project_name, dataset_name, version]

    return '-'.join(to_kebab(part) for part in parts)


fine_tuning_response = client.fine_tuning.jobs.create(
    training_file=training_file_upload_response.id,
    validation_file=validation_file_upload_response.id,
    suffix=generate_finetune_suffix('artefacts', 'met', 'v3'),
    model="gpt-4o-2024-08-06"

)

fine_tuning_response

FineTuningJob(id='ftjob-lI4XwtzHox3S9Qtb8XDx3zIj', created_at=1763466243, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=835290295, status='validating_files', trained_tokens=None, training_file='file-DEuQU4JbtLbg8CJDDedHbU', validation_file='file-TGr42f3AcZ2FnKNbsYZD5d', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='artefacts-met-v3', usage_metrics=None, shared_with_openai=False, eval_id=None)

check training job status

In [ ]:
status_response = client.fine_tuning.jobs.retrieve(fine_tuning_response.id)

status_response

FineTuningJob(id='ftjob-lI4XwtzHox3S9Qtb8XDx3zIj', created_at=1763466243, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4o-2024-08-06', object='fine_tuning.job', organization_id='org-QVEfkKafjj7yW8YeqB6EfSev', result_files=[], seed=835290295, status='running', trained_tokens=None, training_file='file-DEuQU4JbtLbg8CJDDedHbU', validation_file='file-TGr42f3AcZ2FnKNbsYZD5d', estimated_finish=1763470256, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix='artefacts-met-v3', usage_metrics=None, shared_with_openai=False, eval_id=None)

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="sk-xxxxx")
status = client.fine_tuning.jobs.retrieve('ftjob-lI4XwtzHox3S9Qtb8XDx3zIj')
status.status

'succeeded'

run inference using fine-tuned model

In [ ]:
from openai import OpenAI

uzh_data_dir = "/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/description_only"

client = OpenAI(api_key="sk-xxxx")
status_response = client.fine_tuning.jobs.retrieve('ftjob-lI4XwtzHox3S9Qtb8XDx3zIj')
fine_tuned_model = status_response.fine_tuned_model

In [ ]:
fine_tuned_model

'ft:gpt-4o-2024-08-06:university-of-zurich-department-of-history:artefacts-met-v3:CdFToJDn'

In [ ]:
import json
from openai import OpenAI
import re

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl(f"{met_data_dir}/test.jsonl")

completion = client.chat.completions.create(
    model=status_response.fine_tuned_model,
    messages=test_data[0]['messages'][:-1]
)

completion.choices[0].message

ChatCompletionMessage(content='Material: Glass\nCulture: Greek, Eastern Mediterranean\nCategory: Glass', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [ ]:
import json
from openai import OpenAI
import re
from tqdm import tqdm

# Load JSONL file
def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Helper to parse metadata into a dict
def parse_metadata(text):
    """
    Converts text like:
    'Material: Ton\nKultur: Korinthisch\nKategorie: Gefäss'
    into {'Material': 'Ton', 'Kultur': 'Korinthisch', 'Kategorie': 'Gefäss'}
    """
    meta = {}
    for line in text.split("\n"):
        if ':' in line:
            key, val = line.split(":", 1)
            meta[key.strip()] = val.strip()
    return meta

def generate_metadata_predictions(data, model, max_samples=None):
    predictions = []

     # Calculate total_samples
    total_samples = len(data) if max_samples is None else min(max_samples, len(data))

    for i, sample in enumerate(tqdm(data, total=total_samples, desc="Generating metadata")):
        if max_samples and i >= max_samples:
            break

        messages = sample["messages"]

        # Ground truths
        ground_metadata = messages[2]["content"]  # assistant metadata

        # First image URL
        first_image_url = None
        for c in messages[1]["content"]:
            if c.get("type") == "image_url":
                first_image_url = c["image_url"]["url"]
                break

        # Predict metadata
        conversation = [
            messages[0],  # system
            messages[1],  # user asking for metadata
        ]

        completion = client.chat.completions.create(
            model=model,
            messages=conversation
        )
        prediction_metadata = completion.choices[0].message.content.strip()

        # Compare metadata fields
        gt_meta_dict = parse_metadata(ground_metadata)
        pred_meta_dict = parse_metadata(prediction_metadata)
        metadata_match = []
        for key in ["Material", "Culture", "Category"]:
            match = "✅" if gt_meta_dict.get(key, "").lower() == pred_meta_dict.get(key, "").lower() else "❌"
            metadata_match.append(f"{key} {match}")
        metadata_match_str = ", ".join(metadata_match)

        # Save predictions
        predictions.append({
            "sample_id": i + 1,
            "first_image_url": first_image_url,
            "ground_metadata": ground_metadata,
            "prediction_metadata": prediction_metadata,
            "metadata_match": metadata_match_str,
        })

    return predictions

In [ ]:
# Load test set

test_data = load_jsonl(f"{met_data_dir}/test.jsonl")

# Generate predictions
test_preds = generate_metadata_predictions(test_data, fine_tuned_model)

# Print nicely
def print_predictions(preds):
    for r in preds:
        print(json.dumps(r, indent=2, ensure_ascii=False))

print("\n=== Test Set Predictions ===")
print_predictions(test_preds)

Generating metadata: 100%|██████████| 50/50 [03:20<00:00,  4.02s/it]


=== Test Set Predictions ===
{
  "sample_id": 1,
  "first_image_url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255268_primary.jpg",
  "ground_metadata": "Material: Glass\nCulture: Greek, South Italian, Apulian\nCategory: Glass",
  "prediction_metadata": "Material: Glass\nCulture: Greek, Eastern Mediterranean\nCategory: Glass",
  "metadata_match": "Material ✅, Culture ❌, Category ✅"
}
{
  "sample_id": 2,
  "first_image_url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/254675_primary.jpg",
  "ground_metadata": "Material: Terracotta\nCulture: Etruscan\nCategory: Vases",
  "prediction_metadata": "Material: Terracotta\nCulture: Greek, Attic\nCategory: Vases",
  "metadata_match": "Material ✅, Culture ❌, Category ✅"
}
{
  "sample_id": 3,
  "first_image_url": "https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/248796_primary.jpg",
  "ground_metadata": "Material: Terracotta\nCulture: Greek, Attic\nCategory: Vases

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

# Metadata metrics
def compute_metadata_metrics(predictions):
    gt_material, gt_culture, gt_category = [], [], []
    pred_material, pred_culture, pred_category = [], [], []

    for p in predictions:
        # parse ground truth
        gt = parse_metadata(p['ground_metadata'])
        pred = parse_metadata(p['prediction_metadata'])

        gt_material.append(gt.get("Material", "").lower())
        gt_culture.append(gt.get("Culture", "").lower())
        gt_category.append(gt.get("Category", "").lower())

        pred_material.append(pred.get("Material", "").lower())
        pred_culture.append(pred.get("Culture", "").lower())
        pred_category.append(pred.get("Category", "").lower())

    # Compute accuracy, precision, recall, f1 for each field
    def compute_metrics(gt_list, pred_list, label):
        acc = accuracy_score(gt_list, pred_list)
        prec = precision_score(gt_list, pred_list, average='macro', zero_division=0)
        rec = recall_score(gt_list, pred_list, average='macro', zero_division=0)
        f1 = f1_score(gt_list, pred_list, average='macro', zero_division=0)
        print(f"--- {label} ---\nAccuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}\n")

    compute_metrics(gt_material, pred_material, "Material")
    compute_metrics(gt_culture, pred_culture, "Culture")
    compute_metrics(gt_category, pred_category, "Category")

# Run metrics
compute_metadata_metrics(test_preds)

--- Material ---
Accuracy: 0.9000, Precision: 0.4107, Recall: 0.4286, F1: 0.4184

--- Culture ---
Accuracy: 0.5200, Precision: 0.1777, Recall: 0.2202, F1: 0.1866

--- Category ---
Accuracy: 0.9200, Precision: 0.7333, Recall: 0.7750, F1: 0.7516



## Test with un-fine-tuned gpt-4o-2024-08-06 model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from openai import OpenAI
import re
from tqdm import tqdm

In [ ]:
# initiate openai client

from openai import OpenAI

client = OpenAI(api_key="sk-xxx")

### zero-shot-learning

In [ ]:
def generate_metadata_response(message):
    # Create a copy of the message without assistant response
    messages_for_inference = [
        {
            "role": "system",
            "content": "You are a museum curator who creates accurate metadata for archaeological artifacts. Respond ONLY in the exact format: 'Material: X\nCulture: Y\nCategory: Z' without any additional text."
        },
        message["messages"][1]   # user message
    ]

    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=messages_for_inference,
        max_tokens=100,  # metadata is short
        temperature=0.1   # Lower temperature for more consistent formatting
    )
    return response.choices[0].message.content

def process_test_set(test_messages):
    results = []

    for test_message in tqdm(test_messages):
        # Generate response using the test message format
        response = generate_metadata_response(test_message)

        # Extract first image URL
        first_image_url = None
        for content_item in test_message["messages"][1]["content"]:
            if content_item.get("type") == "image_url":
                first_image_url = content_item["image_url"]["url"]
                break

        # Store results
        results.append({
            "original_message": test_message,
            "prediction": response,
            "ground_truth": test_message["messages"][2]["content"],
            "first_image_url": first_image_url
        })

    return results

In [ ]:
# Load test set
with open("/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only_495/test.jsonl", 'r', encoding='utf-8') as f:
    all_test_messages = [json.loads(line.strip()) for line in f]

# Test with first example
test_subset = all_test_messages[:5]
results = process_test_set(test_subset)

# Check results - CORRECTED VERSION
for i, result in enumerate(results):
    print(f"\n--- Sample {i+1} ---")
    print("PREDICTION:")
    print(result['prediction'])
    print("\nGROUND TRUTH:")
    print(result['ground_truth'])
    print(f"\nFirst Image URL: {result['first_image_url']}")
    print("---" * 20)

100%|██████████| 5/5 [00:11<00:00,  2.37s/it]


--- Sample 1 ---
PREDICTION:
Material: Glass  
Culture: Roman  
Category: Bead

GROUND TRUTH:
Material: Glass
Culture: Greek, South Italian, Apulian
Category: Glass

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255268_primary.jpg
------------------------------------------------------------

--- Sample 2 ---
PREDICTION:
Material: Ceramic  
Culture: Greek  
Category: Vessel

GROUND TRUTH:
Material: Terracotta
Culture: Etruscan
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/254675_primary.jpg
------------------------------------------------------------

--- Sample 3 ---
PREDICTION:
Material: Ceramic  
Culture: Greek  
Category: Vase

GROUND TRUTH:
Material: Terracotta
Culture: Greek, Attic
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/248796_primary.jpg
------------------------------------------------------------

--- Sample 4 

### few-shot-learning

In [ ]:
import random
import json
from tqdm import tqdm
from openai import BadRequestError
import time

In [ ]:
def generate_metadata_response_few_shot(message, few_shot_examples, max_retries=3):
    messages_for_inference = [
        {
            "role": "system",
            "content": "You are a museum curator who creates accurate metadata for archaeological artifacts. Respond ONLY in the exact format: 'Material: X\nCulture: Y\nCategory: Z' without any additional text."
        },
        *few_shot_examples,  # Include the few-shot examples
        message["messages"][1]   # current user message
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-2024-08-06",
                messages=messages_for_inference,
                max_tokens=100,
                temperature=0.1
            )
            return response.choices[0].message.content

        except BadRequestError as e:
            if "invalid_image_url" in str(e) and attempt < max_retries - 1:
                print(f"Image URL error, retrying {attempt + 1}/{max_retries}...")
                time.sleep(2)  # Wait before retrying
                continue
            else:
                print(f"Failed after {max_retries} attempts: {e}")
                return f"ERROR: {e}"
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Error {type(e).__name__}, retrying {attempt + 1}/{max_retries}...")
                time.sleep(2)
                continue
            else:
                print(f"Failed after {max_retries} attempts: {e}")
                return f"ERROR: {e}"

In [ ]:
def process_test_set_few_shot(test_messages, few_shot_examples):
    results = []
    failed_samples = []

    for test_message in tqdm(test_messages):
        # Generate response using FEW-SHOT function
        response = generate_metadata_response_few_shot(test_message, few_shot_examples)

        # Extract first image URL
        first_image_url = None
        for content_item in test_message["messages"][1]["content"]:
            if content_item.get("type") == "image_url":
                first_image_url = content_item["image_url"]["url"]
                break

        # Store results
        result_data = {
            "original_message": test_message,
            "prediction": response,
            "ground_truth": test_message["messages"][2]["content"],
            "first_image_url": first_image_url
        }

        # Check if this was an error response
        if isinstance(response, str) and response.startswith("ERROR:"):
            failed_samples.append(result_data)
            print(f"Failed to process sample with image: {first_image_url}")
        else:
            results.append(result_data)

    return results, failed_samples

In [ ]:
# Load test set
with open("/content/drive/My Drive/master_thesis_project/gpt4o/met_split_data/classification_only_495/test.jsonl", 'r', encoding='utf-8') as f:
    all_test_messages = [json.loads(line.strip()) for line in f]

# Process with retry logic
results, failed_samples = process_test_set_few_shot(all_test_messages, few_shot_examples)

print(f"\nProcessing complete!")
print(f"Successfully processed: {len(results)} samples")
print(f"Failed samples: {len(failed_samples)} samples")

# Check successful results
for i, result in enumerate(results[:5]):  # Show first 5 successful results
    print(f"\n--- Sample {i+1} ---")
    print("PREDICTION:")
    print(result['prediction'])
    print("\nGROUND TRUTH:")
    print(result['ground_truth'])
    print(f"\nFirst Image URL: {result['first_image_url']}")
    print("---" * 20)

# Show failed samples if any
if failed_samples:
    print(f"\nFailed samples ({len(failed_samples)}):")
    for failed in failed_samples:
        print(f"Image URL: {failed['first_image_url']}")
        print(f"Error: {failed['prediction']}")

100%|██████████| 50/50 [03:05<00:00,  3.72s/it]


Processing complete!
Successfully processed: 50 samples
Failed samples: 0 samples

--- Sample 1 ---
PREDICTION:
Material: Faience
Culture: Egyptian
Category: Jewelry

GROUND TRUTH:
Material: Glass
Culture: Greek, South Italian, Apulian
Category: Glass

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255268_primary.jpg
------------------------------------------------------------

--- Sample 2 ---
PREDICTION:
Material: Terracotta
Culture: Greek, Apulian
Category: Vases

GROUND TRUTH:
Material: Terracotta
Culture: Etruscan
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/254675_primary.jpg
------------------------------------------------------------

--- Sample 3 ---
PREDICTION:
Material: Terracotta
Culture: Greek, Attic
Category: Vases

GROUND TRUTH:
Material: Terracotta
Culture: Greek, Attic
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolv

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

def parse_metadata(text):
    """
    Converts text like:
    'Material: Ton\nKultur: Korinthisch\nKategorie: Gefäss'
    into {'Material': 'Ton', 'Kultur': 'Korinthisch', 'Kategorie': 'Gefäss'}
    """
    meta = {}
    if not text or text.startswith(("ERROR:", "SKIPPED:")):
        return meta

    for line in text.split("\n"):
        if ':' in line:
            key, val = line.split(":", 1)
            meta[key.strip()] = val.strip()
    return meta

def compute_metadata_metrics(predictions):
    gt_material, gt_culture, gt_category = [], [], []
    pred_material, pred_culture, pred_category = [], [], []

    successful_predictions = []

    for p in predictions:
        # Skip failed predictions
        if isinstance(p['prediction'], str) and p['prediction'].startswith(("ERROR:", "SKIPPED:")):
            continue

        successful_predictions.append(p)

        # parse ground truth and prediction
        gt = parse_metadata(p['ground_truth'])
        pred = parse_metadata(p['prediction'])

        gt_material.append(gt.get("Material", "").lower())
        gt_culture.append(gt.get("Culture", "").lower())
        gt_category.append(gt.get("Category", "").lower())

        pred_material.append(pred.get("Material", "").lower())
        pred_culture.append(pred.get("Culture", "").lower())
        pred_category.append(pred.get("Category", "").lower())

    print(f"Total samples: {len(predictions)}")
    print(f"Successful predictions: {len(successful_predictions)}")
    print(f"Failed predictions: {len(predictions) - len(successful_predictions)}")

    if len(successful_predictions) == 0:
        print("No successful predictions to evaluate!")
        return

    # Compute accuracy, precision, recall, f1 for each field
    def compute_metrics(gt_list, pred_list, label):
        # For macro averaging, we need to handle the case where there might be only one class
        unique_labels = set(gt_list + pred_list)
        if len(unique_labels) == 1:
            # If only one class, accuracy is the only meaningful metric
            acc = accuracy_score(gt_list, pred_list)
            print(f"--- {label} ---\nAccuracy: {acc:.4f} (Only one class present)\n")
        else:
            acc = accuracy_score(gt_list, pred_list)
            prec = precision_score(gt_list, pred_list, average='macro', zero_division=0)
            rec = recall_score(gt_list, pred_list, average='macro', zero_division=0)
            f1 = f1_score(gt_list, pred_list, average='macro', zero_division=0)
            print(f"--- {label} ---\nAccuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}\n")

    compute_metrics(gt_material, pred_material, "Material")
    compute_metrics(gt_culture, pred_culture, "Culture")
    compute_metrics(gt_category, pred_category, "Category")

    # Additional: Exact match accuracy (all three fields correct)
    exact_matches = 0
    for i in range(len(gt_material)):
        if (gt_material[i] == pred_material[i] and
            gt_culture[i] == pred_culture[i] and
            gt_category[i] == pred_category[i]):
            exact_matches += 1

    exact_match_accuracy = exact_matches / len(gt_material)
    print(f"--- Overall ---\nExact Match Accuracy: {exact_match_accuracy:.4f} ({exact_matches}/{len(gt_material)})")

# Run metrics
compute_metadata_metrics(results)

Total samples: 50
Successful predictions: 50
Failed predictions: 0
--- Material ---
Accuracy: 0.8200, Precision: 0.3294, Recall: 0.3202, F1: 0.3195

--- Culture ---
Accuracy: 0.3600, Precision: 0.1273, Recall: 0.0890, F1: 0.1010

--- Category ---
Accuracy: 0.6000, Precision: 0.3235, Recall: 0.2297, F1: 0.2642

--- Overall ---
Exact Match Accuracy: 0.2200 (11/50)


results of fine-tuned model for comparison

--- Material ---
Accuracy: 0.9000, Precision: 0.4107, Recall: 0.4286, F1: 0.4184

--- Culture ---
Accuracy: 0.5200, Precision: 0.1777, Recall: 0.2202, F1: 0.1866

--- Category ---
Accuracy: 0.9200, Precision: 0.7333, Recall: 0.7750, F1: 0.7516

In [ ]:
# Check successful results
for i, result in enumerate(results):  # Show all successful results
    print(f"\n--- Sample {i+1} ---")
    print("PREDICTION:")
    print(result['prediction'])
    print("\nGROUND TRUTH:")
    print(result['ground_truth'])
    print(f"\nFirst Image URL: {result['first_image_url']}")
    print("---" * 20)


--- Sample 1 ---
PREDICTION:
Material: Faience
Culture: Egyptian
Category: Jewelry

GROUND TRUTH:
Material: Glass
Culture: Greek, South Italian, Apulian
Category: Glass

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/255268_primary.jpg
------------------------------------------------------------

--- Sample 2 ---
PREDICTION:
Material: Terracotta
Culture: Greek, Apulian
Category: Vases

GROUND TRUTH:
Material: Terracotta
Culture: Etruscan
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/254675_primary.jpg
------------------------------------------------------------

--- Sample 3 ---
PREDICTION:
Material: Terracotta
Culture: Greek, Attic
Category: Vases

GROUND TRUTH:
Material: Terracotta
Culture: Greek, Attic
Category: Vases

First Image URL: https://huggingface.co/datasets/Huan3/met_artefacts_images/resolve/main/248796_primary.jpg
---------------------------------------------------------